# PortoFlow — khôi phục báo cáo từ Output đã lưu

Chạy notebook này để tiếp tục hậu xử lý. **Không chạy lại matching6giờ.**

Hai Inputs: **code Dataset từ portoflow_report_recovery.zip** và **Notebook Output
Porto_flow vừa lỗi** (anhthu128/porto-flow, chọn đúng saved version). Output có thể
là thư mục hoặc `_output_.zip`5,48GB. Không cần tải file lớn về laptop, raw Porto,
code Dataset cũ, GPU hoặc API credentials. CPU/None, InternetOn.

Gói đọc trực tiếp các member cần thiết trong Output ZIP và kiểm SHA từng batch.
Mã matching/scientific package gốc giữ nguyên; công cụ recovery nằm riêng trong scripts.
Không dùng lại SQLite tổng hợp cũ thiếu receipt; tạo ledger mới có checkpoint,
Arrow-filter trước khi decode và bulk insert; cache từng chunk profile và batch holdout.
Giữ support3/2/2, cutoff, QC và công thức màu/ETA. Unknown giữ nguyên.

Cap report-only90phút, RAM mềm12.000MiB/output12GiB; setup/probes có cap riêng
trong code. Đây là trần, không phải lời hứa thời gian toàn bộ. Nếu PARTIAL, giữ
Output recovery và tiếp tục bằng RESUME trong lần report-only tiếp theo.

Output nhỏ để gửi: **portoflow_recovered_review.zip**. Giữ toàn bộ thư mục
**portoflow_recovery** để bảo toàn ledger/cache/candidate. Công cụ công bố riêng
`matching_complete`: report xong không có nghĩa matching đã xử lý hết parent.
Candidate vẫn chờ review source/QC/holdout; không tự cập nhật app.


In [ ]:

from pathlib import Path
import hashlib,json,os,shutil,signal,subprocess,sys,time,traceback,zipfile

INPUT=Path('/kaggle/input')
PROJECT=Path('/kaggle/working/portoflow_report_code')
WORK=Path('/kaggle/working/portoflow_recovery')
WORK.mkdir(parents=True,exist_ok=True)
EXPECTED={'configs/default.yaml': '87e67d6d2e606cd9679d2b818cfc93f3722dfe168efb966c31102c427533115a', 'configs/study_area.yaml': '3898f37ece0e204ddf94b521cc4b6850fe4161db4d9f29bf064a43de964984e3', 'requirements/constraints.txt': '461ae5985f94aae9d02d5da3552ff09e4f18eed578c5334dfecee65d90ca6537', 'scripts/report_recovery.py': 'c24d11ad11917b2bee742e8eb114aaf51b42c69cc2127ec4c355f5a1eb254263', 'scripts/run_pipeline.py': 'bf2a31813722edf870ca14deb7b8e3a9db86d69574f6ad4dec0a2ce5f2bc05e8', 'src/portoflow/__init__.py': 'ce68b7aaf2f0f077063031d7206e185230d57ff852c2af2c802d959aa6de955a', 'src/portoflow/config.py': '6e1963fdbe6ecd5e8d7420000e3a3c30e4dd72b1d53676b1fdb9e0c375c1d8cc', 'src/portoflow/congestion.py': '58930df59d669afb14154f8d1ed65d4e7c0fa1a52bf4c02ba68091f26bda14d4', 'src/portoflow/contracts.py': 'ef321f5829d847bbd8f704ac847f01cda9ded4fa1860d390a65fb5d5459e7ee2', 'src/portoflow/data.py': '889caafaeb02e137325a799b996e18779d6f4743eca88d8ebcf228a4f47ee762', 'src/portoflow/export.py': '951348ce1383c6e8e750bb829c60d29d8d147e7778ff01a7af6c8c33b012e55b', 'src/portoflow/graph.py': 'ea5b2597ae17ef57f4393863bf1cf4e1486ee6da1c9fdd6a789a8ba53e829bb7', 'src/portoflow/matching.py': 'df5780da53e57564abd4d451fe698c001f08b82a83a12f0efa90e8766b9a95d3', 'src/portoflow/pipeline.py': 'eb187fc2de9a751ddd4af910a784a16c93e7d8f1eb4cf0166e4a9c81d836136c', 'src/portoflow/preprocess.py': 'd7ec5c19cce02d8ea3e7d415333bec42fc25edd0242d5bb468a59449e534db5d', 'src/portoflow/profiles.py': 'f282d3f5fb11ef8910caf76df19120e8ffb4cf9cf4199d149b5d46ef94bb2c93', 'src/portoflow/route.py': '27729173359cf2f32d0f320dc40e02783bcb3c331b99e5b3e33c416f0fb0512a', 'src/portoflow/schema_v1.json': '28b20399945a89ecc930339b4572d1681604cd61ec3052ceae74ded6bdac9b42', 'src/portoflow/schemas.py': '7696d61e82bebaac381960d4c66aa8d15d44a84177ec2c666b7e05f531279b16', 'src/portoflow/segment_qc.py': 'e2613fb6c79b76ff78fd7f2c1e920731829cbbb1fa24b4aadd48b6a410032268', 'src/portoflow/speed.py': '5bd01482a89cf61bbe2723874a5d272724b534eb898acf9f97661b74c0144ede', 'src/portoflow/state.py': '6d80e53ce56074b66dbde98c054328fe277205bfab3d5785914435c46ada405d', 'src/portoflow/storage.py': 'e937952c364a15af437f58e0007662b786c29d91ab990728f572a148197ac07d', 'src/portoflow/synthetic.py': '57f8dac3bc881d077ffe213d5ec753b4411a0e23f082993b078a29e35a53081a', 'tests/test_report_recovery.py': 'dccc0468ae32c069769ef68064bfdd03e0ef72c177f626d9e9279e09d5c23de4'}
SOURCE=None  # Optional explicit Path to the ORIGINAL saved Output directory or _output_.zip.
RESUME=None  # Optional previous recovery Output directory/archive, only if recovery returned PARTIAL.
PROCESSING_SECONDS=5400  # 90-minute report-only cap; no scan or native matching.
environment=dict(os.environ,PYTHONPATH=str(PROJECT/'src'),PYTHONUNBUFFERED='1',
    MPLBACKEND='Agg',MPLCONFIGDIR=str(WORK/'matplotlib_cache'),
    UV_PYTHON_INSTALL_DIR='/tmp/portoflow_recovery_python',UV_CACHE_DIR='/tmp/portoflow_recovery_uv',
    OMP_NUM_THREADS='2',OPENBLAS_NUM_THREADS='2',MKL_NUM_THREADS='2')

def restore_code():
    roots=[p.parent.parent for p in INPUT.rglob('report_recovery.py') if p.parent.name=='scripts']
    for root in roots:
        if all((root/n).is_file() and hashlib.sha256((root/n).read_bytes()).hexdigest()==h for n,h in EXPECTED.items()):
            for name in EXPECTED:
                target=PROJECT/name;target.parent.mkdir(parents=True,exist_ok=True);shutil.copyfile(root/name,target)
            return
    archives=list(INPUT.rglob('portoflow_report_recovery.zip'))
    if len(archives)!=1:
        raise ValueError('Attach the NEW portoflow_report_recovery code Dataset')
    with zipfile.ZipFile(archives[0]) as archive:
        for name,checksum in EXPECTED.items():
            data=archive.read('portoflow_report_recovery/'+name)
            if hashlib.sha256(data).hexdigest()!=checksum:
                raise ValueError('Recovery code checksum differs: '+name)
            target=PROJECT/name;target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(data)

def select_source():
    excluded={'test_tmp','native_probe','report_recovery_synthetic_tests'}
    candidates=[p for p in INPUT.rglob('matching_summary.json') if p.parent.name=='portoflow_run' and not any(s in excluded for s in p.relative_to(INPUT).parts)]
    if len(candidates)==1:
        return candidates[0].parent
    archives=[]
    for path in INPUT.rglob('*.zip'):
        if path.name=='portoflow_report_recovery.zip':
            continue
        if any(s in excluded for s in path.relative_to(INPUT).parts):
            continue
        try:
            with zipfile.ZipFile(path) as archive:
                if any(n.endswith('portoflow_run/matching_summary.json') and not any(s in excluded for s in Path(n).parts) for n in archive.namelist()):
                    archives.append(path)
        except zipfile.BadZipFile:
            continue
    if len(archives)!=1:
        raise ValueError('Attach the ORIGINAL Porto_flow saved Notebook Output; select SOURCE explicitly if ambiguous')
    return archives[0]

def restore_cache(path):
    if path is None or (WORK/'recovery_evidence.sqlite').exists():
        return
    path=Path(path)
    skipped={'candidate_export','evaluation_scratch','matplotlib_cache'}
    if path.is_dir():
        if not (path/'recovery_evidence.sqlite').is_file():
            roots=[p.parent for p in path.rglob('recovery_evidence.sqlite') if p.parent.name=='portoflow_recovery']
            if len(roots)!=1:
                raise ValueError('Select the previous RECOVERY cache, not the original matching database')
            path=roots[0]
        files=[p for p in path.rglob('*') if p.is_file() and p.relative_to(path).parts[0] not in skipped]
        if sum(p.stat().st_size for p in files)>12*1024**3:
            raise ValueError('Recovery cache exceeds12GiB')
        for item in files:
            target=WORK/item.relative_to(path);target.parent.mkdir(parents=True,exist_ok=True);shutil.copyfile(item,target)
    else:
        with zipfile.ZipFile(path) as archive:
            roots=[n[:-len('recovery_evidence.sqlite')] for n in archive.namelist()
                   if n.endswith('portoflow_recovery/recovery_evidence.sqlite')]
            if len(roots)!=1:
                raise ValueError('Select the previous RECOVERY Output archive')
            prefix=roots[0]
            members=[i for i in archive.infolist() if i.filename.startswith(prefix) and not i.is_dir()]
            if sum(i.file_size for i in members)>12*1024**3:
                raise ValueError('Recovery cache exceeds12GiB')
            for item in members:
                raw_relative=item.filename[len(prefix):]
                relative=Path(raw_relative)
                if relative.is_absolute() or '..' in relative.parts or ':' in raw_relative or '\\' in raw_relative:
                    raise ValueError('Unsafe recovery cache path')
                if relative.parts[0] in skipped:
                    continue
                target=WORK/relative;target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(archive.read(item))

def bundle():
    destination=Path('/kaggle/working/portoflow_recovered_review.zip')
    with zipfile.ZipFile(destination,'w',zipfile.ZIP_DEFLATED,compresslevel=1) as archive:
        for path in WORK.rglob('*'):
            if not path.is_file() or path.stat().st_size>16*1024**2:
                continue
            relative=path.relative_to(WORK)
            if relative.parts[0]=='candidate_export':
                if path.name in {'congestion_profiles.parquet','pipeline_config.json','diagnostics.png','run_metadata.json'}:
                    archive.write(path,relative.as_posix())
                continue
            if relative.parts[0] not in {'logs','review_cases'} and len(relative.parts)>1:
                continue
            if path.suffix in {'.json','.txt','.png'}:
                archive.write(path,relative.as_posix())
    print('SEND:',destination,flush=True)
    print('KEEP FULL recovery Output:',WORK,flush=True)
    print('Original matching Output is read-only; no scan/matching launched.',flush=True)

def run(command,name,seconds):
    logs=WORK/'logs';logs.mkdir(exist_ok=True)
    print('START',name,'| cap',seconds,'s',flush=True)
    with (logs/name).open('w') as stream:
        process=subprocess.Popen(list(map(str,command)),cwd=PROJECT,env=environment,
            stdout=stream,stderr=subprocess.STDOUT,start_new_session=True)
        start=time.monotonic();offset=0
        try:
            while process.poll() is None:
                time.sleep(5)
                with (logs/name).open(errors='replace') as reader:
                    reader.seek(offset);text=reader.read();offset=reader.tell()
                print(text[-12000:] if text else name+' | running '+str(round(time.monotonic()-start))+'s',flush=True)
                if time.monotonic()-start>seconds:
                    raise TimeoutError(name+' budget; retain committed recovery cache')
            with (logs/name).open(errors='replace') as reader:
                reader.seek(offset);print(reader.read()[-12000:],flush=True)
            if process.returncode:
                raise RuntimeError(name+' failed; send recovery review ZIP and retain Output')
        finally:
            if process.poll() is None:
                os.killpg(process.pid,signal.SIGKILL);process.wait()

restore_code()
(WORK/'recovery_code_revisions.json').write_text(json.dumps(EXPECTED,indent=2))
if SOURCE is None:
    SOURCE=select_source()
restore_cache(RESUME)
print('ORIGINAL SAVED OUTPUT:',SOURCE,flush=True)
print('REPORT ONLY | 0 raw scan | 0 native matching | checkpointed report progress',flush=True)


In [ ]:

try:
    run([sys.executable,'-m','pip','install','uv>=0.8,<1'],'setup_uv.txt',180)
    uv=shutil.which('uv')
    if not uv:
        raise RuntimeError('uv unavailable')
    run([uv,'python','install','3.12'],'setup_python.txt',600)
    venv=Path('/tmp/portoflow_report_venv')
    if not (venv/'bin/python').is_file():
        run([uv,'venv','--python','3.12',venv],'setup_venv.txt',120)
    python=venv/'bin/python'
    run([uv,'pip','install','--python',python,'-c',PROJECT/'requirements/constraints.txt',
         'PyYAML','pyarrow','networkx','shapely','pyproj','numpy','matplotlib','tzdata','pytest'],
        'setup_report_dependencies.txt',900)
    run([python,'-m','pytest','tests/test_report_recovery.py','-q',
         '--basetemp','/kaggle/working/report_recovery_synthetic_tests'], 'synthetic_recovery_tests.txt',240)
    run([python,PROJECT/'scripts/report_recovery.py','--source',SOURCE,'--work',WORK,
         '--budget-seconds',PROCESSING_SECONDS],'recover_report.txt',PROCESSING_SECONDS+120)
    result=json.loads((WORK/'result_summary.json').read_text())
    print('RECOVERY RESULT:',result['status'],flush=True)
    if result['status']=='REPORT_RECOVERY_PARTIAL':
        print('Keep recovery Output; next report-only run can resume its cache.',flush=True)
except Exception:
    (WORK/'BLOCKER.txt').write_text(traceback.format_exc())
    raise
finally:
    bundle()
